# BG-EN Bitext Matching with Sentence Transformers
Fine-tuning `paraphrase-multilingual-MiniLM-L12-v2` on the OPUS-100 Bulgarian-English dataset using MultipleNegativesRanking (MNR) loss.

## 1. Setup

In [1]:
import sys
print(sys.executable)

c:\Users\raian\source\repos\AI\.env\Scripts\python.exe


In [2]:
from datasets import load_dataset
from transformers import AutoModel, AutoTokenizer
import torch
import torch.nn as nn
import torch.nn.functional as F

checkpoint = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

dataset = load_dataset('Helsinki-NLP/opus-100', 'bg-en')
test_dataset  = dataset['test']['translation']
train_dataset = dataset['train']['translation']
val_dataset   = dataset['validation']['translation']

tokenizer = AutoTokenizer.from_pretrained(checkpoint)

c:\Users\raian\source\repos\AI\.env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda


c:\Users\raian\source\repos\AI\.env\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\raian\.cache\huggingface\hub\datasets--Helsinki-NLP--opus-100. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Generating validation split: 100%|██████████| 2000/2000 [00:00<00:00, 165003.40 examples/s]
c:\Users\raian\so

## 2. Shared Utilities

In [ ]:
def mean_pooling(model_output, attention_mask):
    """Average token embeddings, ignoring padding tokens."""
    token_embeddings = model_output.last_hidden_state  # [B, seq_len, hidden_dim]
    input_mask = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return torch.sum(token_embeddings * input_mask, dim=1) / torch.clamp(input_mask.sum(1), min=1e-9)


def loss_mnr(bg_emb, en_emb, temp=0.07):
    """MultipleNegativesRanking loss. Diagonal of similarity matrix = correct pairs."""
    similarity_matrix = (bg_emb @ en_emb.T) / temp  # [B, B]
    labels = torch.arange(len(bg_emb)).to(bg_emb.device)
    return F.cross_entropy(similarity_matrix, labels)


@torch.no_grad()
def evaluate(model, tokenizer, test_data, device, batch_size=256):
    """Bitext retrieval accuracy: fraction of BG sentences matched to correct EN."""
    model.eval()
    bg_sentences = [x['bg'] for x in test_data]
    en_sentences = [x['en'] for x in test_data]

    def encode(sentences):
        enc = tokenizer(sentences, padding=True, truncation=True, return_tensors='pt')
        enc = {k: v.to(device) for k, v in enc.items()}
        out = model(**enc)
        emb = mean_pooling(out, enc['attention_mask'])
        return F.normalize(emb, p=2, dim=-1)

    bg_emb = encode(bg_sentences)
    en_emb = encode(en_sentences)

    similarity_matrix = bg_emb @ en_emb.T
    indices = torch.argmax(similarity_matrix, dim=1)
    acc = (indices == torch.arange(len(bg_emb), device=device)).float().mean().item()
    print(f"Bitext Retrieval Accuracy: {acc:.4f}")
    return acc

## 3. Baseline Evaluation (frozen model)

In [ ]:
base_model = AutoModel.from_pretrained(checkpoint).to(device)
print("Baseline:", end=' ')
baseline_acc = evaluate(base_model, tokenizer, test_dataset, device)

---
## 4. Manual Training Loop

In [ ]:
from torch.utils.data import DataLoader
from torch.optim import AdamW
from tqdm import tqdm

def collate_fn(batch):
    """Dynamic padding — pads only to the longest sequence in the batch."""
    bg_texts, en_texts = zip(*batch)
    bg_enc = tokenizer(list(bg_texts), padding=True, truncation=True, return_tensors='pt')
    en_enc = tokenizer(list(en_texts), padding=True, truncation=True, return_tensors='pt')
    return bg_enc, en_enc


def make_dataset(data):
    """Simple list of (bg, en) tuples for the manual DataLoader."""
    return [(item['bg'], item['en']) for item in data]

In [ ]:
manual_model = AutoModel.from_pretrained(checkpoint).to(device)

loader = DataLoader(
    make_dataset(train_dataset),
    batch_size=64,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=4,
    pin_memory=True,
)

optim = AdamW(manual_model.parameters(), lr=5e-5, weight_decay=0.01)
num_epochs = 2
manual_model.train()

for epoch in range(num_epochs):
    loop = tqdm(loader, desc=f"Epoch {epoch}")
    for step, (bg_enc, en_enc) in enumerate(loop):
        bg_enc = {k: v.to(device) for k, v in bg_enc.items()}
        en_enc = {k: v.to(device) for k, v in en_enc.items()}

        bg_out = manual_model(**bg_enc)
        en_out = manual_model(**en_enc)

        bg_emb = F.normalize(mean_pooling(bg_out, bg_enc['attention_mask']), p=2, dim=-1)
        en_emb = F.normalize(mean_pooling(en_out, en_enc['attention_mask']), p=2, dim=-1)

        loss = loss_mnr(bg_emb, en_emb)
        loss.backward()
        optim.step()
        optim.zero_grad()

        loop.set_postfix(loss=loss.item())

        if step > 0 and step % 1000 == 0:
            acc = evaluate(manual_model, tokenizer, test_dataset, device)
            manual_model.train()

print("\nManual training done. Final evaluation:")
evaluate(manual_model, tokenizer, test_dataset, device)

---
## 5. Trainer-based Training

In [ ]:
from torch.utils.data import Dataset
from transformers import TrainingArguments, Trainer


class BiTextModel(nn.Module):
    def __init__(self, checkpoint):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(checkpoint)

    def encode(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        emb = mean_pooling(out, attention_mask)
        return F.normalize(emb, p=2, dim=-1)

    def forward(self, bg_input_ids, bg_attention_mask, en_input_ids, en_attention_mask, **kwargs):
        bg_emb = self.encode(bg_input_ids, bg_attention_mask)
        en_emb = self.encode(en_input_ids, en_attention_mask)
        loss = loss_mnr(bg_emb, en_emb)
        return (loss,)  # Trainer expects a tuple


class TranslationDataset(Dataset):
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        return self.data[index]['bg'], self.data[index]['en']


class BiTextCollator:
    """Dynamic padding collator for Trainer — equivalent to the manual collate_fn."""
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer

    def __call__(self, batch):
        bg_texts, en_texts = zip(*batch)
        bg = self.tokenizer(list(bg_texts), padding=True, truncation=True, return_tensors='pt')
        en = self.tokenizer(list(en_texts), padding=True, truncation=True, return_tensors='pt')
        return {
            'bg_input_ids':      bg['input_ids'],
            'bg_attention_mask': bg['attention_mask'],
            'en_input_ids':      en['input_ids'],
            'en_attention_mask': en['attention_mask'],
        }

In [ ]:
trainer_model = BiTextModel(checkpoint).to(device)

args = TrainingArguments(
    output_dir='./models',
    per_device_train_batch_size=64,
    per_device_eval_batch_size=64,
    learning_rate=5e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    num_train_epochs=2,
    eval_strategy='steps',
    eval_steps=1000,
    save_strategy='steps',
    save_steps=1000,
    save_total_limit=2,
    dataloader_num_workers=4,
    dataloader_pin_memory=True,
)

trainer = Trainer(
    model=trainer_model,
    args=args,
    train_dataset=TranslationDataset(train_dataset),
    eval_dataset=TranslationDataset(val_dataset),
    data_collator=BiTextCollator(tokenizer),
)

trainer.train()

In [ ]:
print("Trainer model final evaluation:")
evaluate(trainer_model.encoder, tokenizer, test_dataset, device)